# Simple Medical Image Diffusion with MONAI 🩺

The `stable_diffusion_intro.ipynb` notebook in this folder shows how Stable Diffusion works for **natural images** (astronauts, horses, etc.) using Hugging Face `diffusers`.

But for **medical images** (CT, MRI, X-ray) there is a library built specifically for the job:

> **MONAI Generative** — part of [Project MONAI](https://monai.io/), the standard PyTorch-based framework for medical imaging deep learning. It gives us the same diffusion building blocks (a U-Net + a noise scheduler) but tuned for medical data, with no text prompts needed.

This notebook is intentionally **as simple as possible**: we train a tiny diffusion model that learns to *generate* medical images from scratch, using the small built-in **MedNIST** dataset.

**What you will see:**
1. Install MONAI
2. Download a tiny medical image dataset (MedNIST)
3. Build a diffusion U-Net + scheduler (the two core pieces of Stable Diffusion)
4. Train for a few epochs
5. Generate brand-new synthetic medical images from pure noise

## 1. Install the libraries

The diffusion building blocks (`DiffusionModelUNet`, `DDPMScheduler`) were **merged into MONAI core** in MONAI 1.4 (late 2024), so we just install **core MONAI** — no separate package needed. This is the version Colab and Kaggle already ship, so it's the most reliable on those platforms.

`einops` is required by MONAI's attention/transformer blocks; `tqdm` gives us progress bars.

> **Older alternative:** the standalone `monai-generative` package (frozen at v0.2.3, Jan 2024) still installs, but it breaks against recent MONAI. If you ever need it, the imports become `from generative.networks.nets import ...`. Prefer the core path below.

In [ ]:
# Core MONAI (>=1.4) already includes the generative models. Works on Colab & Kaggle.
!pip install -q "monai>=1.4.0" einops tqdm matplotlib

In [ ]:
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

# Use a GPU if one is available — diffusion training is much faster on GPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Get a tiny medical image dataset (MedNIST)

MONAI can download **MedNIST** for us automatically. It is a small collection of 64×64 medical images across 6 classes (HeadCT, Hand, ChestCT, etc.) — perfect for a quick, easy demo.

To keep things simple we pick **one class** (`HeadCT`) so the model only has to learn one kind of image.

In [ ]:
import os
from monai.apps import MedNISTDataset
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    Resized,
)

root_dir = "./mednist_data"
os.makedirs(root_dir, exist_ok=True)

# Transforms: load image -> add channel dim -> scale pixels to [0, 1] -> resize to 64x64
transforms = Compose(
    [
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys=["image"]),
        ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
        Resized(keys=["image"], spatial_size=(64, 64)),
    ]
)

train_data = MedNISTDataset(root_dir=root_dir, transform=transforms, section="training", download=True)

# Keep only the 'HeadCT' class to keep the task simple.
head_ct = [item for item in train_data.data if item["class_name"] == "HeadCT"]
print(f"Number of HeadCT training images: {len(head_ct)}")

In [ ]:
from monai.data import Dataset, DataLoader

# Wrap the filtered list in a Dataset + DataLoader so we can train in batches.
train_ds = Dataset(data=head_ct, transform=transforms)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)

# Quick look at a few real images
batch = next(iter(train_loader))
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(batch["image"][i, 0], cmap="gray")
    ax.axis("off")
fig.suptitle("Real HeadCT images")
plt.show()

## 3. Build the diffusion model

Just like in the Stable Diffusion intro, a diffusion model has two core pieces:

| Piece | Job |
|-------|-----|
| **U-Net** (`DiffusionModelUNet`) | Looks at a noisy image and predicts the noise that was added |
| **Scheduler** (`DDPMScheduler`) | Knows how much noise to add at each timestep, and how to remove it step by step |

Unlike text-to-image Stable Diffusion, here there is **no text encoder and no prompt** — the model just learns what HeadCT images look like.

In [ ]:
# Imports come from MONAI core now (these moved out of the old 'generative' package).
from monai.networks.nets import DiffusionModelUNet
from monai.networks.schedulers import DDPMScheduler

# A small 2D U-Net: 1 input channel (grayscale), 1 output channel (predicted noise).
model = DiffusionModelUNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    num_channels=(64, 128, 128),
    attention_levels=(False, True, True),
    num_res_blocks=1,
    num_head_channels=128,
).to(device)

# The scheduler defines the noising/denoising process over 1000 timesteps.
scheduler = DDPMScheduler(num_train_timesteps=1000)

optimizer = torch.optim.Adam(model.parameters(), lr=2.5e-4)
print("Model ready.")

## 4. Train

The training loop is the heart of diffusion, and it is surprisingly short:

1. Take a real image.
2. Add a random amount of noise to it.
3. Ask the U-Net to predict the noise that was added.
4. Compare the prediction to the real noise (MSE loss) and update the model.

A few epochs is enough to start seeing recognizable shapes. Increase `n_epochs` for better results.

In [ ]:
import torch.nn.functional as F

n_epochs = 25  # bump this up (e.g. 75) for sharper results if you have time/GPU

model.train()
for epoch in range(n_epochs):
    epoch_loss = 0
    for batch in train_loader:
        images = batch["image"].to(device)
        optimizer.zero_grad()

        # 1. Sample random noise and a random timestep for each image.
        noise = torch.randn_like(images)
        timesteps = torch.randint(0, scheduler.num_train_timesteps, (images.shape[0],), device=device).long()

        # 2. Add that noise to the images according to the scheduler.
        noisy_images = scheduler.add_noise(original_samples=images, noise=noise, timesteps=timesteps)

        # 3. Ask the model to predict the noise, and 4. score it.
        noise_pred = model(x=noisy_images, timesteps=timesteps)
        loss = F.mse_loss(noise_pred, noise)

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{n_epochs} - loss: {epoch_loss / len(train_loader):.4f}")

## 5. Generate new medical images from pure noise

Now the fun part. We start from **pure random noise** and let the scheduler + U-Net remove the noise step by step (1000 steps) until a brand-new, synthetic HeadCT image appears. None of these images existed in the dataset — the model imagined them.

In [ ]:
model.eval()

# Start from pure noise: 4 images, 1 channel, 64x64.
sample = torch.randn((4, 1, 64, 64)).to(device)

scheduler.set_timesteps(num_inference_steps=1000)
with torch.no_grad():
    for t in tqdm(scheduler.timesteps, desc="Denoising"):
        noise_pred = model(x=sample, timesteps=torch.tensor([t], device=device))
        sample, _ = scheduler.step(noise_pred, t, sample)

# Show the generated images.
sample = torch.clamp(sample, 0, 1)
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.imshow(sample[i, 0].cpu(), cmap="gray")
    ax.axis("off")
fig.suptitle("Synthetic HeadCT images generated by the model")
plt.show()

## Recap & where to go next

What we did:
- Used **MONAI Generative**, the medical-imaging-focused diffusion library, instead of generic `diffusers`.
- Trained a small **diffusion U-Net + DDPM scheduler** — the same two core ideas behind Stable Diffusion — on real HeadCT images.
- Generated **new synthetic medical images** from pure noise.

**Ideas to extend this (for the research assistant log):**
- Train on a different MedNIST class (Hand, ChestCT) or on your own DICOM/NIfTI scans via MONAI transforms.
- Add a **Latent Diffusion** stage (`AutoencoderKL`) to work at higher resolution — this is the "latent" part of *Stable* Diffusion.
- Add **conditioning** (e.g. class label or segmentation mask) to control what gets generated.
- Use the official MONAI tutorials: https://github.com/Project-MONAI/GenerativeModels/tree/main/tutorials